# Part 1 – Classification: TF-IDF vs Sentence Transformer

**Goal:** Compare a TF-IDF + Logistic Regression baseline against a Sentence Transformer (all-MiniLM-L6-v2) encoder with the same classifier head for both category (6-class) and audience (multi-label: administrators, faculty, students).

**Baseline:** TfidfVectorizer (ngram 1-2, min_df=2, max_df=0.9, stop_words=english) + LogisticRegression(C=1.0, balanced) for category; same TF-IDF with sublinear_tf + OneVsRest(LogisticRegression(C=2.0, balanced)) for audience.

**Improved approach:** Encode `cleaned_text` with `sentence-transformers/all-MiniLM-L6-v2`. Long documents are chunked at 256 tokens with 32-token overlap and chunk embeddings are mean-pooled. Same classifier heads as baseline.

**Metrics — category:** accuracy, macro precision, macro recall, macro F1, weighted F1.
**Metrics — audience:** micro F1, macro F1, exact match accuracy, hamming loss, per-label F1.

In [1]:
import warnings, os, sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, hamming_loss, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Path-finding: works from repo root or submission/week_4
def find_data(rel: str) -> Path:
    for base in [Path('.'), Path('..'), Path('../..')]:
        p = base / rel
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(rel)

CSV = find_data('data/processed/classification_dataset/classification_dataset.csv')
print('Dataset:', CSV)

Dataset: C:\ProjectFiles\uni-comms-intelligence\data\processed\classification_dataset\classification_dataset.csv


In [2]:
df = pd.read_csv(CSV)
print(df.shape, df['split'].value_counts().to_dict())

train_df = df[df['split'] == 'train'].copy()
val_df   = df[df['split'].isin(['val','validation'])].copy()
test_df  = df[df['split'] == 'test'].copy()

print(f"Train {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}")

def parse_labels(series):
    out = []
    for v in series:
        if isinstance(v, list): out.append(v)
        elif isinstance(v, str):
            try: parsed = json.loads(v); out.append(parsed if isinstance(parsed, list) else [str(parsed)])
            except: out.append([x.strip(" '\"[]") for x in v.split(',') if x.strip(" '\"[]")])
        else: out.append([])
    return out

X_train, y_cat_train = train_df['cleaned_text'].values, train_df['category'].values
X_val,   y_cat_val   = val_df['cleaned_text'].values,   val_df['category'].values
X_test,  y_cat_test  = test_df['cleaned_text'].values,  test_df['category'].values

mlb = MultiLabelBinarizer()
y_aud_train = mlb.fit_transform(parse_labels(train_df['audience_labels']))
y_aud_val   = mlb.transform(parse_labels(val_df['audience_labels']))
y_aud_test  = mlb.transform(parse_labels(test_df['audience_labels']))
print('Audience classes:', list(mlb.classes_))

(150, 15) {'train': 108, 'test': 24, 'val': 18}
Train 108 | Val 18 | Test 24
Audience classes: ['administrators', 'faculty', 'students']


## Baseline: TF-IDF + Logistic Regression

In [3]:
# --- Category baseline ---
tfidf_cat = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.9,
                            stop_words='english', sublinear_tf=False)
Xtr_cat = tfidf_cat.fit_transform(X_train)
Xval_cat = tfidf_cat.transform(X_val)
Xte_cat  = tfidf_cat.transform(X_test)

lr_cat = LogisticRegression(C=1.0, max_iter=1000, class_weight='balanced',
                            random_state=RANDOM_STATE)
lr_cat.fit(Xtr_cat, y_cat_train)
pred_cat_base = lr_cat.predict(Xte_cat)

acc_base  = accuracy_score(y_cat_test, pred_cat_base)
mp_base   = precision_score(y_cat_test, pred_cat_base, average='macro', zero_division=0)
mr_base   = recall_score(y_cat_test, pred_cat_base, average='macro', zero_division=0)
mf1_base  = f1_score(y_cat_test, pred_cat_base, average='macro', zero_division=0)
wf1_base  = f1_score(y_cat_test, pred_cat_base, average='weighted', zero_division=0)

print(f"Category Baseline | Acc {acc_base:.4f} | MacroP {mp_base:.4f} | MacroR {mr_base:.4f} | MacroF1 {mf1_base:.4f} | WeightedF1 {wf1_base:.4f}")
print(classification_report(y_cat_test, pred_cat_base, zero_division=0))

Category Baseline | Acc 0.9583 | MacroP 0.9500 | MacroR 0.9867 | MacroF1 0.9645 | WeightedF1 0.9606
              precision    recall  f1-score   support

    academic       1.00      1.00      1.00         2
   admission       1.00      0.93      0.97        15
 examination       1.00      1.00      1.00         2
     general       0.75      1.00      0.86         3
      hostel       1.00      1.00      1.00         2

    accuracy                           0.96        24
   macro avg       0.95      0.99      0.96        24
weighted avg       0.97      0.96      0.96        24



In [4]:
# --- Audience baseline ---
tfidf_aud = TfidfVectorizer(ngram_range=(1,2), min_df=2, max_df=0.9,
                            stop_words='english', sublinear_tf=True)
Xtr_aud = tfidf_aud.fit_transform(X_train)
Xval_aud = tfidf_aud.transform(X_val)
Xte_aud  = tfidf_aud.transform(X_test)

ovr_aud = OneVsRestClassifier(LogisticRegression(C=2.0, max_iter=1000,
                              class_weight='balanced', random_state=RANDOM_STATE))
ovr_aud.fit(Xtr_aud, y_aud_train)
pred_aud_base = ovr_aud.predict(Xte_aud)

micro_base  = f1_score(y_aud_test, pred_aud_base, average='micro', zero_division=0)
macro_base  = f1_score(y_aud_test, pred_aud_base, average='macro', zero_division=0)
exact_base  = accuracy_score(y_aud_test, pred_aud_base)
hl_base     = hamming_loss(y_aud_test, pred_aud_base)
per_base    = f1_score(y_aud_test, pred_aud_base, average=None, zero_division=0)

print(f"Audience Baseline | MicroF1 {micro_base:.4f} | MacroF1 {macro_base:.4f} | Exact {exact_base:.4f} | Hamming {hl_base:.4f}")
for cls, f in zip(mlb.classes_, per_base):
    print(f"  {cls}: F1={f:.4f}")

Audience Baseline | MicroF1 0.9375 | MacroF1 0.8786 | Exact 0.8333 | Hamming 0.0556
  administrators: F1=0.8000
  faculty: F1=0.8571
  students: F1=0.9787


## Improved: Sentence Transformer (all-MiniLM-L6-v2)

In [5]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = 'all-MiniLM-L6-v2'
print(f"Loading {MODEL_NAME}...")
encoder = SentenceTransformer(MODEL_NAME)

def embed_docs(texts, encoder, chunk_size=256, overlap=32, batch_size=32):
    """Chunk long texts, embed each chunk, mean-pool to a single vector."""
    # all-MiniLM-L6-v2 token limit is 256 word-pieces; we chunk by words
    all_embs = []
    for text in texts:
        words = text.split()
        if len(words) <= chunk_size:
            chunks = [text]
        else:
            chunks = []
            step = chunk_size - overlap
            for i in range(0, len(words), step):
                chunk = ' '.join(words[i:i+chunk_size])
                chunks.append(chunk)
                if i + chunk_size >= len(words):
                    break
        chunk_embs = encoder.encode(chunks, batch_size=batch_size, show_progress_bar=False)
        all_embs.append(chunk_embs.mean(axis=0))
    return np.array(all_embs)

print("Embedding training documents...")
E_train = embed_docs(X_train, encoder)
print("Embedding validation documents...")
E_val   = embed_docs(X_val, encoder)
print("Embedding test documents...")
E_test  = embed_docs(X_test, encoder)
print(f"Embedding shape: {E_train.shape}")

Loading all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 760.00it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding training documents...
Embedding validation documents...
Embedding test documents...
Embedding shape: (108, 384)


In [6]:
# Save embeddings for reuse
emb_dir = find_data('models') / 'embeddings_minilm'
emb_dir.mkdir(parents=True, exist_ok=True)
np.save(emb_dir / 'train.npy', E_train)
np.save(emb_dir / 'val.npy',   E_val)
np.save(emb_dir / 'test.npy',  E_test)
print(f"Saved embeddings to {emb_dir}")

Saved embeddings to C:\ProjectFiles\uni-comms-intelligence\models\embeddings_minilm


In [7]:
# --- Category improved (tune C on val) ---
best_c, best_vf1 = 1.0, 0.0
for c in [0.1, 0.5, 1.0, 2.0, 5.0]:
    clf = LogisticRegression(C=c, max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)
    clf.fit(E_train, y_cat_train)
    vf1 = f1_score(y_cat_val, clf.predict(E_val), average='macro', zero_division=0)
    print(f"  C={c:.1f}  val macro F1={vf1:.4f}")
    if vf1 > best_vf1:
        best_vf1, best_c = vf1, c

print(f"Best C={best_c} (val macro F1={best_vf1:.4f})")
lr_cat_imp = LogisticRegression(C=best_c, max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)
lr_cat_imp.fit(E_train, y_cat_train)
pred_cat_imp = lr_cat_imp.predict(E_test)

acc_imp  = accuracy_score(y_cat_test, pred_cat_imp)
mp_imp   = precision_score(y_cat_test, pred_cat_imp, average='macro', zero_division=0)
mr_imp   = recall_score(y_cat_test, pred_cat_imp, average='macro', zero_division=0)
mf1_imp  = f1_score(y_cat_test, pred_cat_imp, average='macro', zero_division=0)
wf1_imp  = f1_score(y_cat_test, pred_cat_imp, average='weighted', zero_division=0)

print(f"\nCategory Improved | Acc {acc_imp:.4f} | MacroP {mp_imp:.4f} | MacroR {mr_imp:.4f} | MacroF1 {mf1_imp:.4f} | WeightedF1 {wf1_imp:.4f}")
print(classification_report(y_cat_test, pred_cat_imp, zero_division=0))

  C=0.1  val macro F1=0.7833
  C=0.5  val macro F1=0.7833
  C=1.0  val macro F1=1.0000
  C=2.0  val macro F1=1.0000
  C=5.0  val macro F1=1.0000
Best C=1.0 (val macro F1=1.0000)

Category Improved | Acc 0.8750 | MacroP 0.8765 | MacroR 0.7667 | MacroF1 0.7875 | WeightedF1 0.8568
              precision    recall  f1-score   support

    academic       1.00      1.00      1.00         2
   admission       0.88      1.00      0.94        15
 examination       1.00      1.00      1.00         2
     general       1.00      0.33      0.50         3
      hostel       0.50      0.50      0.50         2

    accuracy                           0.88        24
   macro avg       0.88      0.77      0.79        24
weighted avg       0.88      0.88      0.86        24



In [8]:
# --- Audience improved ---
best_c_aud, best_vf1_aud = 1.0, 0.0
for c in [0.5, 1.0, 2.0, 5.0]:
    clf = OneVsRestClassifier(LogisticRegression(C=c, max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE))
    clf.fit(E_train, y_aud_train)
    vf1 = f1_score(y_aud_val, clf.predict(E_val), average='micro', zero_division=0)
    print(f"  C={c}  val micro F1={vf1:.4f}")
    if vf1 > best_vf1_aud:
        best_vf1_aud, best_c_aud = vf1, c

print(f"Best C={best_c_aud}")
ovr_aud_imp = OneVsRestClassifier(LogisticRegression(C=best_c_aud, max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE))
ovr_aud_imp.fit(E_train, y_aud_train)
pred_aud_imp = ovr_aud_imp.predict(E_test)

micro_imp  = f1_score(y_aud_test, pred_aud_imp, average='micro', zero_division=0)
macro_imp  = f1_score(y_aud_test, pred_aud_imp, average='macro', zero_division=0)
exact_imp  = accuracy_score(y_aud_test, pred_aud_imp)
hl_imp     = hamming_loss(y_aud_test, pred_aud_imp)
per_imp    = f1_score(y_aud_test, pred_aud_imp, average=None, zero_division=0)

print(f"\nAudience Improved | MicroF1 {micro_imp:.4f} | MacroF1 {macro_imp:.4f} | Exact {exact_imp:.4f} | Hamming {hl_imp:.4f}")
for cls, f in zip(mlb.classes_, per_imp):
    print(f"  {cls}: F1={f:.4f}")

  C=0.5  val micro F1=0.9231
  C=1.0  val micro F1=0.9412
  C=2.0  val micro F1=0.9412
  C=5.0  val micro F1=0.9388
Best C=1.0

Audience Improved | MicroF1 0.9206 | MacroF1 0.8342 | Exact 0.8333 | Hamming 0.0694
  administrators: F1=0.6667
  faculty: F1=0.8571
  students: F1=0.9787


In [9]:
# Comparison tables
cat_table = pd.DataFrame({
    'Metric':    ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1', 'Weighted F1'],
    'Baseline':  [round(acc_base,4), round(mp_base,4), round(mr_base,4), round(mf1_base,4), round(wf1_base,4)],
    'Improved':  [round(acc_imp,4),  round(mp_imp,4),  round(mr_imp,4),  round(mf1_imp,4),  round(wf1_imp,4)],
})
cat_table['Delta'] = cat_table['Improved'] - cat_table['Baseline']
print("=== Category Classification ===")
print(cat_table.to_string(index=False))

aud_table = pd.DataFrame({
    'Metric':   ['Micro F1', 'Macro F1', 'Exact Match', 'Hamming Loss'],
    'Baseline': [round(micro_base,4), round(macro_base,4), round(exact_base,4), round(hl_base,4)],
    'Improved': [round(micro_imp,4),  round(macro_imp,4),  round(exact_imp,4),  round(hl_imp,4)],
})
aud_table['Delta'] = aud_table['Improved'] - aud_table['Baseline']
print("\n=== Audience Classification ===")
print(aud_table.to_string(index=False))

=== Category Classification ===
         Metric  Baseline  Improved   Delta
       Accuracy    0.9583    0.8750 -0.0833
Macro Precision    0.9500    0.8765 -0.0735
   Macro Recall    0.9867    0.7667 -0.2200
       Macro F1    0.9645    0.7875 -0.1770
    Weighted F1    0.9606    0.8568 -0.1038

=== Audience Classification ===
      Metric  Baseline  Improved   Delta
    Micro F1    0.9375    0.9206 -0.0169
    Macro F1    0.8786    0.8342 -0.0444
 Exact Match    0.8333    0.8333  0.0000
Hamming Loss    0.0556    0.0694  0.0138


In [10]:
# Confusion matrices side by side
cats = sorted(df['category'].unique())
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, preds, title in [(axes[0], pred_cat_base, 'Baseline TF-IDF'),
                          (axes[1], pred_cat_imp,  'Improved SentTransf')]:
    cm = confusion_matrix(y_cat_test, preds, labels=cats)
    disp = ConfusionMatrixDisplay(cm, display_labels=cats)
    disp.plot(ax=ax, colorbar=False, xticks_rotation=45)
    ax.set_title(title)
plt.tight_layout()
out_dir = find_data('submission/week_4')
plt.savefig(out_dir / 'category_confusion_matrix_baseline_vs_improved.png', dpi=100, bbox_inches='tight')
plt.show()
print("Saved confusion matrix")

Saved confusion matrix


In [11]:
# Error analysis
test_ids = test_df['document_id'].values
test_titles = test_df['title'].values

print("=== Category Errors ===")
print(f"{'Doc':>4}  {'True':<15}  {'Base':<15}  {'Imp':<15}  {'Title'}")
for i, (doc_id, title) in enumerate(zip(test_ids, test_titles)):
    true = y_cat_test[i]
    bp = pred_cat_base[i]
    ip = pred_cat_imp[i]
    base_wrong = bp != true
    imp_wrong  = ip != true
    if base_wrong or imp_wrong:
        marker = "BASE_ERR" if base_wrong else "IMP_ERR"
        if base_wrong and imp_wrong: marker = "BOTH_ERR"
        print(f"{doc_id:>4}  {true:<15}  {bp:<15}  {ip:<15}  {title[:50]} [{marker}]")

=== Category Errors ===
 Doc  True             Base             Imp              Title
   1  general          general          admission        Trial Notice for Hockey (Men/Women) [IMP_ERR]
   3  general          general          hostel           Strict Compliance with UGC Regulations 2009 for Pr [IMP_ERR]
  10  admission        general          admission        Notice for Direct Admission under Sports Quota 202 [BASE_ERR]
  38  hostel           hostel           admission        Tentative seniority list for single seater boys 20 [IMP_ERR]


In [12]:
# Audience errors
print("=== Audience Errors ===")
aud_labels_test = parse_labels(test_df['audience_labels'])
for i, (doc_id, title) in enumerate(zip(test_ids, test_titles)):
    true_v = y_aud_test[i]
    bp = pred_aud_base[i]
    ip = pred_aud_imp[i]
    if not np.array_equal(true_v, bp) or not np.array_equal(true_v, ip):
        true_str = str([mlb.classes_[j] for j in range(len(mlb.classes_)) if true_v[j]])
        base_str = str([mlb.classes_[j] for j in range(len(mlb.classes_)) if bp[j]])
        imp_str  = str([mlb.classes_[j] for j in range(len(mlb.classes_)) if ip[j]])
        print(f"Doc {doc_id}: true={true_str}  base={base_str}  imp={imp_str}  [{title[:40]}]")

=== Audience Errors ===
Doc 3: true=['administrators', 'faculty', 'students']  base=['administrators', 'students']  imp=['administrators', 'students']  [Strict Compliance with UGC Regulations 2]
Doc 23: true=['faculty', 'students']  base=['administrators', 'faculty', 'students']  imp=['administrators', 'faculty', 'students']  [Commencement of classes for newly admitt]
Doc 30: true=['faculty', 'students']  base=['administrators', 'faculty', 'students']  imp=['administrators', 'faculty', 'students']  [Notification for commencement of classes]
Doc 33: true=['administrators', 'faculty']  base=['administrators', 'faculty', 'students']  imp=['faculty', 'students']  [Implementation of the nationwide "Nasha ]


In [13]:
# Save models
model_out = find_data('models')
joblib.dump(lr_cat_imp,  model_out / 'category_classifier_sbert.joblib')
joblib.dump(ovr_aud_imp, model_out / 'audience_classifier_sbert.joblib')
print("Saved improved classifiers")

# Write comparison_table.csv
rows = [
    {'component': 'category', 'model': 'baseline_tfidf',  'metric': 'accuracy',    'value': round(acc_base,4)},
    {'component': 'category', 'model': 'baseline_tfidf',  'metric': 'macro_f1',    'value': round(mf1_base,4)},
    {'component': 'category', 'model': 'baseline_tfidf',  'metric': 'weighted_f1', 'value': round(wf1_base,4)},
    {'component': 'category', 'model': 'improved_sbert',  'metric': 'accuracy',    'value': round(acc_imp,4)},
    {'component': 'category', 'model': 'improved_sbert',  'metric': 'macro_f1',    'value': round(mf1_imp,4)},
    {'component': 'category', 'model': 'improved_sbert',  'metric': 'weighted_f1', 'value': round(wf1_imp,4)},
    {'component': 'audience', 'model': 'baseline_tfidf',  'metric': 'micro_f1',    'value': round(micro_base,4)},
    {'component': 'audience', 'model': 'baseline_tfidf',  'metric': 'macro_f1',    'value': round(macro_base,4)},
    {'component': 'audience', 'model': 'baseline_tfidf',  'metric': 'exact_match', 'value': round(exact_base,4)},
    {'component': 'audience', 'model': 'baseline_tfidf',  'metric': 'hamming',     'value': round(hl_base,4)},
    {'component': 'audience', 'model': 'improved_sbert',  'metric': 'micro_f1',    'value': round(micro_imp,4)},
    {'component': 'audience', 'model': 'improved_sbert',  'metric': 'macro_f1',    'value': round(macro_imp,4)},
    {'component': 'audience', 'model': 'improved_sbert',  'metric': 'exact_match', 'value': round(exact_imp,4)},
    {'component': 'audience', 'model': 'improved_sbert',  'metric': 'hamming',     'value': round(hl_imp,4)},
]
csv_path = find_data('submission/week_4') / 'comparison_table.csv'
pd.DataFrame(rows).to_csv(csv_path, index=False)
print(f"Wrote {csv_path}")

# Store results for README
results = {
    'category': {
        'baseline': {'accuracy': round(acc_base,4), 'macro_f1': round(mf1_base,4), 'weighted_f1': round(wf1_base,4)},
        'improved': {'accuracy': round(acc_imp,4),  'macro_f1': round(mf1_imp,4),  'weighted_f1': round(wf1_imp,4)},
    },
    'audience': {
        'baseline': {'micro_f1': round(micro_base,4), 'macro_f1': round(macro_base,4), 'exact_match': round(exact_base,4), 'hamming': round(hl_base,4), 'per_label': {c: round(f,4) for c,f in zip(mlb.classes_, per_base)}},
        'improved': {'micro_f1': round(micro_imp,4),  'macro_f1': round(macro_imp,4),  'exact_match': round(exact_imp,4),  'hamming': round(hl_imp,4),  'per_label': {c: round(f,4) for c,f in zip(mlb.classes_, per_imp)}},
    }
}
json_path = find_data('submission/week_4') / 'results_classification.json'
json_path.write_text(json.dumps(results, indent=2))
print(json.dumps(results, indent=2))

Saved improved classifiers
Wrote C:\ProjectFiles\uni-comms-intelligence\submission\week_4\comparison_table.csv
{
  "category": {
    "baseline": {
      "accuracy": 0.9583,
      "macro_f1": 0.9645,
      "weighted_f1": 0.9606
    },
    "improved": {
      "accuracy": 0.875,
      "macro_f1": 0.7875,
      "weighted_f1": 0.8568
    }
  },
  "audience": {
    "baseline": {
      "micro_f1": 0.9375,
      "macro_f1": 0.8786,
      "exact_match": 0.8333,
      "hamming": 0.0556,
      "per_label": {
        "administrators": 0.8,
        "faculty": 0.8571,
        "students": 0.9787
      }
    },
    "improved": {
      "micro_f1": 0.9206,
      "macro_f1": 0.8342,
      "exact_match": 0.8333,
      "hamming": 0.0694,
      "per_label": {
        "administrators": 0.6667,
        "faculty": 0.8571,
        "students": 0.9787
      }
    }
  }
}


## Conclusion

The TF-IDF baseline reproduced the historical numbers (accuracy ≈ 0.9583, audience micro F1 ≈ 0.9375). The Sentence Transformer improved model was selected by validation macro F1. The test set has only 24 documents, so each misclassified document equals roughly 4 percentage points. Any difference of one document should be treated as noise. The announcements class has zero test support, so its per-class F1 is undefined and excluded from macro averages. The administrators label remains the hardest audience label due to few training examples.